In [ ]:
import os
import subprocess
import time
import threading
import sys
import re
import shutil
import socket
import signal

# -------------------------------------------------------------------------
# 0. Global configuration (your dataset paths)
# -------------------------------------------------------------------------
ROOT_DIR = "/kaggle/working"
COMFY_DIR = os.path.join(ROOT_DIR, "ComfyUI")
MODELS_DIR = os.path.join(COMFY_DIR, "models")

# === Path configuration area (please verify these) ===
FLUX_DATASET_PATH = "/kaggle/input/flux-models/models_download"
WAN_DATASET_PATH  = "/kaggle/input/datasets/grandmachasingthief/wan2-2-models/models_download"
PULID_DATASET_PATH = "/kaggle/input/pulid-models-pack/pulid_models_pack"
LORA_DATASET_PATH = "/kaggle/input/datasets/grandmachasingthief/lora-models"
QWEN_DATASET_PATH = "/kaggle/input/qwen-models/models_download"
LLM_DATASET_PATH = "/kaggle/input/llm-model"
SAM_DATASET_PATH = "/kaggle/input/datasets/grandmachasingthief/sam3-1/models_download"

NGROK_TOKEN = "optional"
os.environ['GROQ_API_KEY'] = 'required'

def run_cmd(cmd, msg=None):
    if msg: print(f"Atara [System]: {msg}")
    try:
        subprocess.run(cmd, shell=True, check=True)
    except subprocess.CalledProcessError as e:
        if "pkill" not in cmd:
            print(f"Atara [Warning]: Command may have failed: {cmd}")

# -------------------------------------------------------------------------
# 1. Infrastructure
# -------------------------------------------------------------------------
print("\n>>> 1. Infrastructure check <<<")
run_cmd("pkill -f main.py || true", "Cleaning up old processes...")
run_cmd("pkill -f ssh || true", "Cleaning up old tunnels...")

if not os.path.exists(COMFY_DIR):
    print("Atara [Builder]: Rebuilding ComfyUI environment...")
    run_cmd("apt-get update -qq && apt-get install -y -qq libgl1 libglib2.0-0 ffmpeg zstd pciutils")

    # 1. Install Ollama
    print("Atara [Builder]: Installing Ollama...")
    run_cmd("curl -fsSL https://ollama.com/install.sh | sh")

    print("Atara [Builder]: Installing ComfyUI-Manager custom nodes...")
    run_cmd(f"git clone https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}")
    run_cmd(f"git clone https://github.com/ltdrdata/ComfyUI-Manager.git {COMFY_DIR}/custom_nodes/ComfyUI-Manager")

    print("Atara [Builder]: Installing ComfyUI-Ollama custom nodes...")
    run_cmd(f"git clone https://github.com/stavsap/ComfyUI-Ollama.git {COMFY_DIR}/custom_nodes/ComfyUI-Ollama")

    print("Atara [Builder]: Installing Impact Pack...")
    run_cmd(f"git clone https://github.com/ltdrdata/ComfyUI-Impact-Pack.git {COMFY_DIR}/custom_nodes/ComfyUI-Impact-Pack")

    print("Atara [Builder]: Installing ComfyUI-KJNodes...")
    run_cmd(f"git clone https://github.com/kijai/ComfyUI-KJNodes.git {COMFY_DIR}/custom_nodes/ComfyUI-KJNodes")

    print("Atara [Builder]: Installing ComfyUI-Crystools (resource monitoring)...")
    run_cmd(f"git clone https://github.com/crystian/ComfyUI-Crystools.git {COMFY_DIR}/custom_nodes/ComfyUI-Crystools")

    # Install base dependencies
    run_cmd(f"pip install -r {os.path.join(COMFY_DIR, 'requirements.txt')}")
    run_cmd(f"pip install -r {COMFY_DIR}/custom_nodes/ComfyUI-Manager/requirements.txt")
    run_cmd(f"pip install -r {COMFY_DIR}/custom_nodes/ComfyUI-Ollama/requirements.txt")
    run_cmd(f"pip install -r {COMFY_DIR}/custom_nodes/ComfyUI-Impact-Pack/requirements.txt")
    run_cmd("pip install comfy-cli torchsde einops transformers safetensors aiohttp accelerate pyyaml opencv-python matplotlib pillow scipy imageio[ffmpeg] moviepy huggingface_hub gguf insightface onnxruntime-gpu")

# -------------------------------------------------------------------------
# 2. Core engine calibration (★ Fix focus ★)
# -------------------------------------------------------------------------
print("\n>>> 2. Core engine calibration (force version alignment) <<<")
if not os.path.exists(COMFY_DIR):
    try:
        import torch
        ver = torch.__version__
        print(f"Atara [Check]: Current PyTorch version -> {ver}")

        # Force reinstall torchvision to match current PyTorch and specify CUDA 12.4 index
        print("Atara [Fix]: Fixing torchvision::nms error...")
        run_cmd(f"pip install --force-reinstall torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124")

        # Also handle xformers
        print("Atara [Fix]: Installing compatible xformers...")
        run_cmd("pip install xformers==0.0.29.post2 --index-url https://download.pytorch.org/whl/cu124 || pip install xformers")

    except Exception as e:
        print(f"Atara [Error]: Encountered a minor issue during fixes ({e}), attempting to continue...")

# -------------------------------------------------------------------------
# 3. Mount user repositories (Checkpoints / PuLID / LoRA)
# -------------------------------------------------------------------------
print("\n>>> 3. Mounting user repositories (Symlink mode) <<<")
print()
# --- A. Base large models and CLIP ---
mapping_list = [
    ("flux1-dev-Q4_0.gguf",         FLUX_DATASET_PATH, "unet"),
    ("t5-v1_1-xxl-encoder-Q5_K_M.gguf", FLUX_DATASET_PATH, "clip"),
    ("clip_l.safetensors",          FLUX_DATASET_PATH, "clip"),
    ("ae.safetensors",              FLUX_DATASET_PATH, "vae"),
    ("Wan2.2-TI2V-5B-Q6_K.gguf",    WAN_DATASET_PATH,  "unet"),
    ("Wan2_2-TI2V-5B-Turbo-Q8_0.gguf",    WAN_DATASET_PATH,  "unet"),
    ("Wan22_TI2V_5B_Turbo_lora_rank_64_fp16.safetensors", WAN_DATASET_PATH, "loras"),
    ("Wan2.2_VAE.safetensors",      WAN_DATASET_PATH,  "vae"),
    ("wan_2.1_vae.safetensors",      WAN_DATASET_PATH,  "vae"),
    ("umt5-xxl-enc-fp8_e4m3fn.safetensors", WAN_DATASET_PATH, "clip"),
    ("umt5_xxl_fp8_e4m3fn_scaled.safetensors", WAN_DATASET_PATH, "clip"),
    ("open-clip-xlm-roberta-large-vit-huge-14_visual_fp16.safetensors",   WAN_DATASET_PATH,  "clip_vision"),
    ("clip_vision_h.safetensors",          WAN_DATASET_PATH, "clip_vision"),
    ("qwen-rapid-nsfw-v9.0-Q4_K_M.gguf",         QWEN_DATASET_PATH, "unet"),
    ("Qwen2.5-VL-7B-Instruct-abliterated.Q4_K_M.gguf", QWEN_DATASET_PATH, "clip"),
    ("Qwen2.5-VL-7B-Instruct-abliterated.mmproj-Q8_0.gguf", QWEN_DATASET_PATH, "clip"),
    ("pig_qwen_image_vae_fp32-f16.gguf", QWEN_DATASET_PATH, "vae"),
    ("sam3.1_multiplex_fp16.safetensors", SAM_DATASET_PATH, "checkpoints"),
]

for filename, source_base, target_sub in mapping_list:
    src = os.path.join(source_base, filename)
    dst_dir = os.path.join(COMFY_DIR, "models", target_sub)
    dst = os.path.join(dst_dir, filename)
    os.makedirs(dst_dir, exist_ok=True)
    if os.path.exists(src):
        if os.path.exists(dst) or os.path.islink(dst): os.remove(dst)
        os.symlink(src, dst)
        print(f"Atara [Link]: ✅ {filename}")
    else:
        print(f"Atara [Warning]: ❌ Cannot find {filename}")

# --- B. PuLID model mount (special handling) ---
print("Atara [Link]: Mounting PuLID & InsightFace...")

# 1. PuLID main model
pulid_src = f"{PULID_DATASET_PATH}/pulid/pulid_flux_v0.9.0.safetensors"
pulid_dst = f"{COMFY_DIR}/models/pulid/pulid_flux_v0.9.0.safetensors"
os.makedirs(os.path.dirname(pulid_dst), exist_ok=True)

if os.path.exists(pulid_src):
    if os.path.exists(pulid_dst) or os.path.islink(pulid_dst): os.remove(pulid_dst)
    os.symlink(pulid_src, pulid_dst)
    print("Atara [Link]: ✅ PuLID Flux Model")
else:
    print(f"Atara [Error]: Cannot find PuLID model, please check PULID_DATASET_PATH: {pulid_src}")

# 2. InsightFace AntelopeV2
insight_src_dir = f"{PULID_DATASET_PATH}/insightface/models/antelopev2"
insight_dst_dir = f"{COMFY_DIR}/models/insightface/models/antelopev2"
os.makedirs(os.path.dirname(insight_dst_dir), exist_ok=True)

if os.path.exists(insight_src_dir):
    if os.path.exists(insight_dst_dir) or os.path.islink(insight_dst_dir):
        if os.path.islink(insight_dst_dir): os.remove(insight_dst_dir)
        else: shutil.rmtree(insight_dst_dir)
    os.symlink(insight_src_dir, insight_dst_dir)
    print("Atara [Link]: ✅ InsightFace AntelopeV2")
else:
    print(f"Atara [Error]: Cannot find InsightFace model")

# --- C. [NEW] LoRA auto-mount logic ---
print(f"Atara [Link]: Scanning LoRA folder: {LORA_DATASET_PATH}")
lora_dst_dir = os.path.join(COMFY_DIR, "models", "loras")
os.makedirs(lora_dst_dir, exist_ok=True)

lora_count = 0
if os.path.exists(LORA_DATASET_PATH):
    for filename in os.listdir(LORA_DATASET_PATH):
        # Scan for all .safetensors files
        if filename.endswith(".safetensors"):
            src = os.path.join(LORA_DATASET_PATH, filename)
            dst = os.path.join(lora_dst_dir, filename)

            # If target exists, remove and relink (ensure latest)
            if os.path.exists(dst) or os.path.islink(dst):
                os.remove(dst)

            os.symlink(src, dst)
            print(f"Atara [LoRA]: 🔗 Link successful: {filename}")
            lora_count += 1

    if lora_count == 0:
        print("Atara [LoRA]: ⚠️ No .safetensors files found in target folder")
else:
    print(f"Atara [LoRA]: ❌ Error: Cannot find path {LORA_DATASET_PATH} (please modify the script header settings)")

# --- D. LLM (Ollama preparation) ---
llm_model_list = []

if os.path.exists(LLM_DATASET_PATH):
    print(f"Atara [Ollama]: Scanning LLM folder...")
    for f in os.listdir(LLM_DATASET_PATH):
        if f.endswith(".gguf"):
            full_path = os.path.join(LLM_DATASET_PATH, f)
            model_name = os.path.splitext(f)[0].lower().replace(" ", "-").replace("_", "-")
            llm_model_list.append((model_name, full_path))
            print(f"Atara [Ollama]: 🎯 Found model: {f} -> planned registration name: '{model_name}'")
    if not llm_model_list:
        print("Atara [Ollama]: ⚠️ No .gguf files found in LLM folder")
else:
    print(f"Atara [Ollama]: ⚠️ Cannot find path {LLM_DATASET_PATH}")

# -------------------------------------------------------------------------
# 4. Start related functions
# -------------------------------------------------------------------------
print("\n>>> 4. Final startup <<<")
run_cmd("pip cache purge")
time.sleep(2)

ollama_process = None
comfyui_process = None
cloudflared_process = None
ngrok_active = False
pinggy_process = None

def start_ollama_service():
    """Start Ollama server and batch register models"""
    global ollama_process
    print("Atara [Ollama]: Starting background service...")
    ollama_process = subprocess.Popen("ollama serve", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    if llm_model_list:
        print(f"Atara [Ollama]: Preparing to register {len(llm_model_list)} models...")
        for name, path in llm_model_list:
            print(f"Atara [Ollama]: Registering '{name}' ...")
            modelfile_content = f"FROM {path}\nPARAMETER num_ctx 4096"
            with open("Modelfile", "w") as f:
                f.write(modelfile_content)
            try:
                subprocess.run(f"ollama create {name} -f Modelfile", shell=True, check=True)
                print(f"\033[1;32mAtara [Ollama]: ✅ Model '{name}' registered successfully!\033[0m")
            except subprocess.CalledProcessError:
                print(f"Atara [Ollama]: ❌ Model '{name}' registration failed")
        print(f"Atara [Ollama]: 🎉 All models processed!")
    else:
        print("Atara [Ollama]: ⚠️ No models to register")

def wait_for_comfyui(port=8188, timeout=120):
    print("Atara [Network]: Waiting for ComfyUI to start...")
    start = time.time()
    while time.time() - start < timeout:
        try:
            s = socket.create_connection(("localhost", port), timeout=2)
            s.close()
            print("Atara [Network]: ComfyUI started, beginning to create Tunnel...")
            return True
        except:
            time.sleep(1)
    print("Atara [Network]: ❌ Waiting for ComfyUI timed out")
    return False

def start_comfyui():
    global comfyui_process
    print("Atara [Backend]: Starting ComfyUI...")
    cmd = "python main.py --listen 0.0.0.0 --port 8188 --preview-method auto"
    comfyui_process = subprocess.Popen(cmd, shell=True, cwd=COMFY_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1, universal_newlines=True)

    for line in iter(comfyui_process.stdout.readline, ''):
        line = line.strip()
        if line:
            if "To see the GUI" in line:
                print(f"\033[1;32m[ComfyUI]: ✅ Service ready! Waiting for connections...\033[0m")
            elif "Error" in line or "Traceback" in line:
                print(f"\033[1;31m[ComfyUI Error]: {line}\033[0m")

def start_pinggy():
    global pinggy_process
    time.sleep(8)
    print("Atara [Network]: Starting Pinggy...")
    cmd = "ssh -o StrictHostKeyChecking=no -p 443 -R0:localhost:8188 a.pinggy.io"
    pinggy_process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1, universal_newlines=True)

    for line in iter(pinggy_process.stdout.readline, ''):
        if "http" in line and "pinggy" in line:
            match = re.search(r'(https?://[^\s]+)', line)
            if match:
                url = match.group(1)
                print("\n==========================================================")
                print(f"   >>> 🚀 Launch URL: {url} 🚀 <<<")
                print("==========================================================\n")

def start_cloudflare():
    """Start Cloudflared in no-login mode (trycloudflare) and print public URL"""
    global cloudflared_process
    print("Atara [Network]: Starting Cloudflare Tunnel...")

    # Download a clean cloudflared and ensure no old config exists
    run_cmd("rm -rf ~/.cloudflared || true", "Clearing old cloudflared configuration")
    run_cmd("rm -f cloudflared || true")
    run_cmd("curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o cloudflared")
    run_cmd("chmod +x cloudflared")

    # Wait for ComfyUI to be fully up
    if not wait_for_comfyui():
        print("Atara [Network]: Cannot detect ComfyUI, Cloudflared will not start")
        return

    # Kaggle stable mode: HTTP/2 + no-autoupdate
    cmd = "./cloudflared tunnel --no-autoupdate --protocol http2 --url http://localhost:8188"
    cloudflared_process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1, universal_newlines=True)

    # Read output and look for trycloudflare URL
    for line in iter(cloudflared_process.stdout.readline, ''):
        line = line.strip()
        if not line:
            continue
        # Print logs for debugging
        print(f"[cloudflared] {line}")
        if "trycloudflare.com" in line or "trycloudflare" in line:
            match = re.search(r'(https?://[^\s]+)', line)
            if match:
                url = match.group(1)
                print("\n==========================================================")
                print(f"   >>> 🌐 Cloudflare public URL: {url} <<<")
                print("==========================================================\n")
                # Once URL is found, continue monitoring without reprinting
                # If auto-reconnect is needed, add monitoring and restart logic here
                # break

def start_ngrok():
    """Start Ngrok (requires NGROK_TOKEN to be set)"""
    global ngrok_active
    print("Atara [Network]: Preparing to start Ngrok Tunnel...")

    try:
        run_cmd("pip install -q pyngrok")
        from pyngrok import ngrok
    except Exception as e:
        print(f"Atara [Error]: Unable to install or import pyngrok: {e}")
        return

    # Wait for ComfyUI to be fully up
    if not wait_for_comfyui():
        print("Atara [Network]: Cannot detect ComfyUI, Ngrok will not start")
        return

    if NGROK_TOKEN == "請把你的_Ngrok_Token_貼在這裡" or not NGROK_TOKEN:
        print("\nAtara [Error]: ❌ You have not filled in the Ngrok Token! Please get one at https://dashboard.ngrok.com/ and put it into the script.\n")
        return

    try:
        ngrok.set_auth_token(NGROK_TOKEN)
        public_url = ngrok.connect(8188).public_url
        ngrok_active = True
        print("\n==========================================================")
        print(f"   >>> 🌟 Ngrok public stable URL: {public_url} <<<")
        print("==========================================================\n")
        print("Atara [Network]: Ngrok tunnel established; free tier may disconnect occasionally.")
    except Exception as e:
        print(f"\nAtara [Error]: ❌ Ngrok failed to start: {e}\n")

# -------------------------------------------------------------------------
# Thread management (fixed, single startup point)
# -------------------------------------------------------------------------
# Choose which tunnel to start (only one will be started)
USE_NGROK = False
USE_CLOUDFLARE = True
USE_PINGGY = False

def start_tunnel_manager():
    try:
        if USE_NGROK:
            start_ngrok()
        elif USE_CLOUDFLARE:
            start_cloudflare()
        elif USE_PINGGY:
            start_pinggy()
        else:
            print("Atara [Network]: No tunnel selected, only starting ComfyUI.")
    except Exception as e:
        print(f"Atara [Network]: Tunnel manager encountered an error: {e}")

# Start ComfyUI and tunnel manager (single startup point)
t_comfy = threading.Thread(target=start_comfyui, name="ComfyUI-Thread", daemon=True)
t_tunnel = threading.Thread(target=start_tunnel_manager, name="Tunnel-Thread", daemon=True)

t_comfy.start()
time.sleep(2)  # Give ComfyUI a moment to start checks
t_tunnel.start()

# Graceful shutdown handling
def _terminate_process(proc, name):
    try:
        if proc and proc.poll() is None:
            proc.terminate()
            print(f"Atara [Cleanup]: terminate sent to {name}")
            # Wait briefly then kill if still alive
            time.sleep(2)
            if proc.poll() is None:
                proc.kill()
                print(f"Atara [Cleanup]: {name} forcibly killed")
    except Exception as e:
        print(f"Atara [Cleanup]: Unable to terminate {name}: {e}")

try:
    while True:
        time.sleep(10)
        if comfyui_process and comfyui_process.poll() is not None:
            print(f"Atara [CRITICAL]: ComfyUI stopped. Code: {comfyui_process.returncode}")
            break

except KeyboardInterrupt:
    print("\nAtara: User interrupt received, starting shutdown...")

finally:
    # Clean up Ngrok
    if ngrok_active:
        try:
            from pyngrok import ngrok
            ngrok.kill()
            print("Atara [Network]: Ngrok closed")
        except Exception:
            pass

    # Terminate cloudflared (if subprocess)
    _terminate_process(cloudflared_process, "cloudflared")

    # Terminate pinggy (if subprocess)
    _terminate_process(pinggy_process, "pinggy")

    # Terminate Ollama
    _terminate_process(ollama_process, "ollama")

    # Terminate ComfyUI
    _terminate_process(comfyui_process, "ComfyUI")

    print("Atara: Attempted to close all processes, exiting.")



>>> 1. Infrastructure check <<<
Atara [System]: Cleaning up old processes...
Atara [System]: Cleaning up old tunnels...
Atara [Builder]: Rebuilding ComfyUI environment...


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


(Reading database ... 129073 files and directories currently installed.)
Preparing to unpack .../0-libglib2.0-dev_2.72.4-0ubuntu2.9_amd64.deb ...
Unpacking libglib2.0-dev:amd64 (2.72.4-0ubuntu2.9) over (2.72.4-0ubuntu2.7) ...
Preparing to unpack .../1-libglib2.0-dev-bin_2.72.4-0ubuntu2.9_amd64.deb ...
Unpacking libglib2.0-dev-bin (2.72.4-0ubuntu2.9) over (2.72.4-0ubuntu2.7) ...
Preparing to unpack .../2-libglib2.0-bin_2.72.4-0ubuntu2.9_amd64.deb ...
Unpacking libglib2.0-bin (2.72.4-0ubuntu2.9) over (2.72.4-0ubuntu2.7) ...
Preparing to unpack .../3-libglib2.0-0_2.72.4-0ubuntu2.9_amd64.deb ...
Unpacking libglib2.0-0:amd64 (2.72.4-0ubuntu2.9) over (2.72.4-0ubuntu2.7) ...
Selecting previously unselected package pci.ids.
Preparing to unpack .../4-pci.ids_0.0~2022.01.22-1ubuntu0.1_all.deb ...
Unpacking pci.ids (0.0~2022.01.22-1ubuntu0.1) ...
Selecting previously unselected package libpci3:amd64.
Preparing to unpack .../5-libpci3_1%3a3.7.0-6_amd64.deb ...
Unpacking libpci3:amd64 (1:3.7.0-6) .

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Cloning into '/kaggle/working/ComfyUI'...


Atara [Builder]: Installing ComfyUI-Manager custom nodes...


...

  Cloning https://github.com/facebookresearch/sam2 to /tmp/pip-req-build-iw70h2or


  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/sam2 /tmp/pip-req-build-iw70h2or


  Resolved https://github.com/facebookresearch/sam2 to commit 2b90b9f5ceec907a1c18123530e92e794ad901a4
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 6.6 MB/s eta 0:00:00
  Created wheel for SAM-2: filename=sam_2-1.0-cp312-cp312-linux_x86_64.whl size=183669 sha256=998072d9bddce6c97ef7ce5adc79f95276c750f7be7bbb71436bda4a593b79a9
  Stored in directory: /tmp/pip-ephem-wheel-cache-t8z5wbsg/wheels/53/42/6e/0cc240e3a26dbb838e1549e1410ebf131571f287c8e592dbc0
  Created wheel for 

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 37.8M  100 37.8M    0     0  31.4M      0  0:00:01  0:00:01 --:--:-- 80.3M


Atara [Network]: Waiting for ComfyUI to start...
[ComfyUI]: ✅ Service ready! Waiting for connections...
Atara [Network]: ComfyUI started, beginning to create Tunnel...
[cloudflared] 2026-05-01T04:16:05Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
[cloudflared] 2026-05-01T04:16:05Z INF Requesting new quick Tunnel on trycloudflare.com...
[cloudflared] 2026-05-01T04:16:08Z INF +-------------------------------------------------------------------------------------

In [4]:
# 1. Uninstall old or incomplete packages
!pip uninstall -y onnxruntime onnxruntime-gpu nvidia-cudnn-cu12 nvidia-cublas-cu12

# 2. Reinstall onnxruntime-gpu with NVIDIA core libraries (this is the key!)
!pip install onnxruntime onnxruntime-gpu --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/
!pip install nvidia-cudnn-cu12 nvidia-cublas-cu12

# 3. Patch the environment to ensure the libraries are discoverable (for Kaggle/Colab)
import os
import sys

# Add installation paths to the environment variable, just in case
for path in sys.path:
    if "site-packages" in path:
        cudnn_path = os.path.join(path, "nvidia/cudnn/lib")
        cublas_path = os.path.join(path, "nvidia/cublas/lib")
        if os.path.exists(cudnn_path):
            os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":" + cudnn_path
        if os.path.exists(cublas_path):
            os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":" + cublas_path

print("✅ Force reinstall complete! Make sure to restart the session.")

Found existing installation: onnxruntime 1.23.2
Uninstalling onnxruntime-1.23.2:
  Successfully uninstalled onnxruntime-1.23.2
Found existing installation: onnxruntime-gpu 1.23.2
Uninstalling onnxruntime-gpu-1.23.2:
  Successfully uninstalled onnxruntime-gpu-1.23.2
Found existing installation: nvidia-cudnn-cu12 9.10.2.21
Uninstalling nvidia-cudnn-cu12-9.10.2.21:
  Successfully uninstalled nvidia-cudnn-cu12-9.10.2.21
Found existing installation: nvidia-cublas-cu12 12.6.4.1
Uninstalling nvidia-cublas-cu12-12.6.4.1:
  Successfully uninstalled nvidia-cublas-cu12-12.6.4.1
Looking in indexes: https://pypi.org/simple, https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 101.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 MB 6.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 648.6/648.6 MB 2.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━

In [12]:
import os

# 檔案路徑
file_path = "/kaggle/working/ComfyUI/custom_nodes/ComfyUI-WanVideoWrapper/multitalk/multitalk_loop.py"

if os.path.exists(file_path):
    print(f"📂 正在讀取檔案: {file_path}")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    is_modified = False

    # --- 🛠️ 修復 1: 第 72 行 (檢查 multitalk_embeds) ---
    target_1 = "if len(multitalk_embeds['audio_features'])==2"
    patch_1 = "if multitalk_embeds is not None and len(multitalk_embeds['audio_features'])==2"
    
    if target_1 in content:
        content = content.replace(target_1, patch_1)
        is_modified = True
        print("✅ [1/3] 修復 multitalk_embeds 檢查。")
    elif patch_1 in content:
        print("ℹ️ [1/3] 已經修復過。")

    # --- 🛠️ 修復 2: 第 95 行 (檢查 audio_embedding) ---
    target_2 = "human_num = len(audio_embedding)"
    patch_2 = "human_num = len(audio_embedding) if audio_embedding is not None else 0"
    
    if target_2 in content:
        content = content.replace(target_2, patch_2)
        is_modified = True
        print("✅ [2/3] 修復 human_num 檢查。")
    elif patch_2 in content:
        print("ℹ️ [2/3] 已經修復過。")

    # --- 🛠️ 修復 3: 第 112 行 (這次的新錯誤！) ---
    # 錯誤原因：total_frames = len(audio_embedding[0])
    # 修復邏輯：如果沒聲音，就把限制設為 100000 (代表不限制，讓它跑完原本設定的幀數)
    target_3 = "total_frames = len(audio_embedding[0])"
    patch_3 = "total_frames = len(audio_embedding[0]) if audio_embedding is not None else 100000"

    if target_3 in content:
        content = content.replace(target_3, patch_3)
        is_modified = True
        print("✅ [3/3] 修復 total_frames 崩潰錯誤。")
    elif patch_3 in content:
        print("ℹ️ [3/3] 已經修復過。")

    # --- 💾 寫回檔案 ---
    if is_modified:
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(content)
        print("\n🎉 V3 補丁寫入完成！請務必 **重啟 ComfyUI** 讓修改生效。")
    else:
        print("\n👌 檔案已經是最新的修復狀態，無需變更。")

else:
    print(f"❌ 找不到檔案，路徑錯誤: {file_path}")

📂 正在讀取檔案: /kaggle/working/ComfyUI/custom_nodes/ComfyUI-WanVideoWrapper/multitalk/multitalk_loop.py
ℹ️ [1/3] 已經修復過。
✅ [2/3] 修復 human_num 檢查。
✅ [3/3] 修復 total_frames 崩潰錯誤。

🎉 V3 補丁寫入完成！請務必 **重啟 ComfyUI** 讓修改生效。


In [15]:
import torch
import onnxruntime as ort

print("------ Status Report ------")

# 1. Check if ComfyUI (PyTorch) can still use the GPU
# If True, the earlier error had no lasting effect
print(f"PyTorch (ComfyUI) GPU status: {'✅ OK' if torch.cuda.is_available() else '❌ Broken'}")

# 2. Check if PuLID (ONNX) can detect the GPU
providers = ort.get_available_providers()
print(f"ONNX (PuLID) available providers: {providers}")

if 'CUDAExecutionProvider' in providers:
    print("\n🎉 Success! Both backends are working. Generation should be fast now!")
else:
    print("\n💀 Still failing... ONNX still cannot detect the GPU.")


------ 檢查報告 ------
PyTorch (ComfyUI) GPU 狀態: ✅ 正常
ONNX (PuLID) 可用裝置: ['AzureExecutionProvider', 'CPUExecutionProvider']

💀 還是失敗... ONNX 依然抓不到 GPU


In [2]:
# Fix the error in newer ComfyUI version
import os
import glob

print("🚀 Fix PuLID-Flux-II (GitHub Issue #86)...")

# 1. Locate the target file pulidflux.py
# Path may vary by environment, so we search recursively to be safe
search_pattern = "/kaggle/working/ComfyUI/custom_nodes/**/pulidflux.py"
files = glob.glob(search_pattern, recursive=True)

if not files:
    print("❌ Cannot find pulidflux.py!")
    print("   Make sure you have 'ComfyUI_PuLID_Flux_ll' (lldacing version) installed.")
else:
    target_file = files[0]
    print(f"✅ Found file: {target_file}")

    # 2. Read and modify the content
    with open(target_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    new_lines = []
    fixed = False

    for line in lines:
        # Look for the function definition that causes the error
        if "def pulid_outer_sample_wrappers_with_override" in line:
            # Check if it has already been patched
            if "latent_shapes" in line:
                print("ℹ️ Code already contains latent_shapes, no fix needed.")
                new_lines.append(line)
            else:
                print("🔧 Applying patch: adding latent_shapes=None parameter...")
                # Per the issue suggestion, append this parameter at the end
                # Original ending is usually seed=None): or disable_pbar=False):
                # We use a general replacement: replace ): with , latent_shapes=None):

                # To avoid replacing mid-line parentheses, we target the line ending
                new_line = line.replace("):", ", latent_shapes=None):")

                # Fallback: if the above didn't match (different format), try replacing seed=None
                if new_line == line:
                     new_line = line.replace("seed=None", "seed=None, latent_shapes=None")

                new_lines.append(new_line)
                fixed = True
        else:
            new_lines.append(line)

    # 3. Save the changes
    if fixed:
        with open(target_file, 'w', encoding='utf-8') as f:
            f.writelines(new_lines)
        print("\n🎉 Patch applied successfully per Issue #86.")
        print("👉 Next step: click 'Restart' in ComfyUI to reload the server!")
    else:
        print("\n⚠️ Scan complete, no changes made (already patched or matching line not found).")


🚀 Fix PuLID-Flux-II (GitHub Issue #86)...
✅ 找到檔案: /kaggle/working/ComfyUI/custom_nodes/comfyui_pulid_flux_ll/pulidflux.py
🔧 正在植入補丁: 增加 latent_shapes=None 參數...

🎉 修復成功！已依照 Issue #86 更新代碼。
👉 請務必執行下一步：點擊 ComfyUI 的 'Restart' 重啟伺服器！
